# Modélisation — Placement optimal de hubs de drones médicaux au Ghana

**Problématique** : Si aucun opérateur de drones médicaux n'existait au Ghana, quel réseau déployer *from scratch* pour maximiser la couverture des facilities de santé en zones rurales ?

**Approche** : Filtrage des facilities desservant les zones rurales, puis K-Means clustering sur leurs coordonnées, avec évaluation de 2 scénarios de déploiement (Coût optimal, Couverture optimale).

---
## 1. Configuration

In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import folium
from folium import plugins
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)

print("Imports OK")

In [ ]:
# --- Constantes métier ---
MAX_RADIUS_KM: float = 80.0
DRONE_SPEED_KMH: float = 110.0

TIER_FACILITY_LIMITS: dict[str, int] = {"Micro": 150, "Standard": 400}
TIER_DRONES: dict[str, tuple[int, int]] = {
    "Micro": (4, 8),
    "Standard": (10, 20),
    "Large": (25, 50),
}
CAPEX_USD: dict[str, int] = {"Micro": 700_000, "Standard": 2_000_000, "Large": 4_500_000}
OPEX_ANNUEL_USD: dict[str, int] = {"Micro": 131_000, "Standard": 311_000, "Large": 728_000}

TIER_COLORS: dict[str, str] = {"Micro": "#00D4AA", "Standard": "#FFA502", "Large": "#FF4757"}

DATA_DIR = Path("..") / "data"
RAW_DIR = DATA_DIR / "Final Group"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Rayon max : {MAX_RADIUS_KM} km")
print(f"Vitesse drone : {DRONE_SPEED_KMH} km/h")

In [ ]:
# --- Fonctions Haversine ---

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distance Haversine entre deux points (km)."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def haversine_matrix(
    lats1: np.ndarray, lons1: np.ndarray,
    lats2: np.ndarray, lons2: np.ndarray,
) -> np.ndarray:
    """Matrice de distances Haversine vectorisée (km)."""
    R = 6371.0
    lat1 = np.radians(lats1)
    lat2 = np.radians(lats2)
    dlat = lat2 - lat1
    dlon = np.radians(lons2) - np.radians(lons1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


print(f"Test : Accra → Kumasi = {haversine_km(5.6037, -0.1870, 6.6885, -1.6244):.1f} km")

In [ ]:
# --- Fonctions utilitaires réutilisables ---

def assign_tier(n_facilities: int) -> str:
    """Type de hub selon le nombre de facilities couvertes."""
    for tier, limit in TIER_FACILITY_LIMITS.items():
        if n_facilities <= limit:
            return tier
    return "Large"


def calc_drones(n_facilities: int, tier: str) -> int:
    """Nombre de drones : max(d_min, ceil(n/20)), plafonné à d_max."""
    d_min, d_max = TIER_DRONES[tier]
    raw = max(d_min, int(np.ceil(n_facilities / 20)))
    return min(raw, d_max)


def compute_coverage(
    centroids: np.ndarray,
    fac_lats: np.ndarray,
    fac_lons: np.ndarray,
    max_km: float = MAX_RADIUS_KM,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Retourne (hub_ids, distances_min, mask_covered) pour chaque facility."""
    f_lats = fac_lats[:, np.newaxis]
    f_lons = fac_lons[:, np.newaxis]
    c_lats = centroids[:, 0][np.newaxis, :]
    c_lons = centroids[:, 1][np.newaxis, :]
    dist = haversine_matrix(f_lats, f_lons, c_lats, c_lons)
    hub_ids = dist.argmin(axis=1)
    dist_min = dist.min(axis=1)
    covered = dist_min <= max_km
    return hub_ids, dist_min, covered


def build_hub_summary(
    centroids: np.ndarray,
    hub_ids: np.ndarray,
    dist_min: np.ndarray,
    covered: np.ndarray,
    fac_pop: np.ndarray,
    force_tier: str | None = None,
    min_tier: str | None = None,
) -> pd.DataFrame:
    """Tableau récapitulatif par hub avec dimensionnement et budget."""
    tier_order = ["Micro", "Standard", "Large"]
    rows = []
    for hid in range(len(centroids)):
        mask = hub_ids == hid
        n_fac = int(mask.sum())
        n_covered = int((mask & covered).sum())
        pop_served = int(fac_pop[mask].sum())
        d_max = float(dist_min[mask].max()) if mask.any() else 0.0
        d_moy = float(dist_min[mask].mean()) if mask.any() else 0.0

        if force_tier:
            tier = force_tier
        elif min_tier:
            natural = assign_tier(n_fac)
            if tier_order.index(natural) < tier_order.index(min_tier):
                tier = min_tier
            else:
                tier = natural
        else:
            tier = assign_tier(n_fac)

        n_drones = calc_drones(n_fac, tier)

        rows.append({
            "Hub_ID": hid,
            "Latitude": round(centroids[hid, 0], 6),
            "Longitude": round(centroids[hid, 1], 6),
            "Type": tier,
            "N_facilities": n_fac,
            "N_covered": n_covered,
            "Pop_desservie": pop_served,
            "Dist_max_km": round(d_max, 2),
            "Dist_moy_km": round(d_moy, 2),
            "N_drones": n_drones,
            "CAPEX_USD": CAPEX_USD[tier],
            "OPEX_annuel_USD": OPEX_ANNUEL_USD[tier],
        })
    return pd.DataFrame(rows)


def network_budget(summary: pd.DataFrame) -> dict:
    """Agrégation budget réseau."""
    capex = int(summary["CAPEX_USD"].sum())
    opex = int(summary["OPEX_annuel_USD"].sum())
    return {
        "N_hubs": len(summary),
        "Repartition": summary["Type"].value_counts().to_dict(),
        "N_drones": int(summary["N_drones"].sum()),
        "CAPEX_USD": capex,
        "OPEX_annuel_USD": opex,
        "TCO_5ans_USD": capex + 5 * opex,
    }


def make_folium_map(
    summary: pd.DataFrame,
    fac_df: pd.DataFrame,
    hub_ids: np.ndarray,
    covered: np.ndarray,
    title: str,
) -> folium.Map:
    """Carte Folium avec hubs, cercles de couverture et facilities."""
    m = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="CartoDB positron")

    for _, row in summary.iterrows():
        color = TIER_COLORS[row["Type"]]
        folium.Circle(
            location=[row["Latitude"], row["Longitude"]],
            radius=MAX_RADIUS_KM * 1000,
            color=color,
            fill=True,
            fill_opacity=0.08,
            weight=1.5,
        ).add_to(m)
        folium.Marker(
            location=[row["Latitude"], row["Longitude"]],
            popup=(
                f"Hub {int(row['Hub_ID'])} ({row['Type']})<br>"
                f"Facilities : {int(row['N_facilities'])}<br>"
                f"Drones : {int(row['N_drones'])}<br>"
                f"Pop : {int(row['Pop_desservie']):,}"
            ),
            icon=folium.Icon(color="black", icon="plane", prefix="fa"),
        ).add_to(m)

    for i in range(len(fac_df)):
        c = "#2ecc71" if covered[i] else "#e74c3c"
        folium.CircleMarker(
            location=[fac_df.iloc[i]["Latitude"], fac_df.iloc[i]["Longitude"]],
            radius=2,
            color=c,
            fill=True,
            fill_opacity=0.7,
            weight=0.5,
        ).add_to(m)

    legend_html = f"""
    <div style="position:fixed; bottom:30px; right:30px; z-index:1000;
                background:white; padding:12px 16px; border-radius:8px;
                box-shadow:0 2px 8px rgba(0,0,0,0.2); font-size:13px;">
        <b>{title}</b><br>
        <i style="color:#00D4AA">&#9632;</i> Micro (4-8 drones)<br>
        <i style="color:#FFA502">&#9632;</i> Standard (10-20 drones)<br>
        <i style="color:#FF4757">&#9632;</i> Large (25-50 drones)<br>
        <i style="color:#2ecc71">&#9679;</i> Facility couverte<br>
        <i style="color:#e74c3c">&#9679;</i> Facility non couverte
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    return m


print("Fonctions utilitaires chargées")

---
## 2. Chargement et préparation des données

In [ ]:
facilities = pd.read_csv(RAW_DIR / "ghana_health_eda_final.csv")
villages = pd.read_csv(RAW_DIR / "ghana_villages_eda_final.csv")

print(f"Facilities : {len(facilities)} lignes, {facilities.shape[1]} colonnes")
print(f"Villages   : {len(villages)} lignes, {villages.shape[1]} colonnes")

# Nettoyage NaN dans les colonnes textuelles (evite erreur Folium)
for col in facilities.select_dtypes(include="object").columns:
    facilities[col] = facilities[col].fillna("")
for col in villages.select_dtypes(include="object").columns:
    villages[col] = villages[col].fillna("")


In [ ]:
facilities.head(3)

In [ ]:
villages.head(3)

In [ ]:
# Population desservie par facility : somme des populations des villages
# dont elle est la facility la plus proche
pop_by_facility = (
    villages
    .groupby("Nearest_Facility_Name")["Population"]
    .sum()
    .reset_index()
    .rename(columns={"Population": "Pop_desservie"})
)

facilities = facilities.merge(
    pop_by_facility,
    left_on="Facility_Name",
    right_on="Nearest_Facility_Name",
    how="left",
).drop(columns=["Nearest_Facility_Name"])

facilities["Pop_desservie"] = facilities["Pop_desservie"].fillna(0).astype(int)

print(f"Facilities avec population renseignée : {(facilities['Pop_desservie'] > 0).sum()} / {len(facilities)}")
print(f"Population totale couverte : {facilities['Pop_desservie'].sum():,}")

In [ ]:
facilities.describe()

In [ ]:
print("Répartition par catégorie :")
print(facilities["Category"].value_counts())

In [ ]:
# Coordonnées pour le clustering
fac_coords = facilities[["Latitude", "Longitude"]].values
fac_lats = facilities["Latitude"].values
fac_lons = facilities["Longitude"].values
fac_pop = facilities["Pop_desservie"].values

print(f"Matrice de coordonnées : {fac_coords.shape}")
print(f"Latitude  : [{fac_lats.min():.2f}, {fac_lats.max():.2f}]")
print(f"Longitude : [{fac_lons.min():.2f}, {fac_lons.max():.2f}]")

---
## 3. Analyse exploratoire spatiale

In [ ]:
# Carte des facilities colorées par catégorie
cat_colors = {
    "CHPS": "#3498db",
    "Clinic": "#2ecc71",
    "Health Centre": "#f39c12",
    "Hospital": "#e74c3c",
    "Pharmacy": "#9b59b6",
    "Maternity Home": "#e67e22",
    "RCH": "#1abc9c",
}

m_eda = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="CartoDB positron")

for _, row in facilities.iterrows():
    cat = row["Category"]
    color = cat_colors.get(cat, "#95a5a6")
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
        weight=0.5,
        popup=f"{row['Facility_Name']} ({cat})",
    ).add_to(m_eda)

legend_items = "".join(
    f'<i style="color:{c}">&#9679;</i> {cat}<br>'
    for cat, c in cat_colors.items()
)
legend_html = f"""
<div style="position:fixed; bottom:30px; right:30px; z-index:1000;
            background:white; padding:12px 16px; border-radius:8px;
            box-shadow:0 2px 8px rgba(0,0,0,0.2); font-size:13px;">
    <b>Facilities par catégorie</b><br>{legend_items}
</div>
"""
m_eda.get_root().html.add_child(folium.Element(legend_html))
m_eda.save(str(PROCESSED_DIR / "carte_facilities_eda.html"))
m_eda

### HeatMap — Densité de population

La carte de chaleur visualise la densite de population des villages ghanéens. Les zones a forte concentration (rouge) sont les bassins de demande prioritaires pour le positionnement des hubs de drones.

In [ ]:
from folium.plugins import HeatMap

m_heat = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="CartoDB positron")

heat_data = [
    [row["Latitude"], row["Longitude"], row["Population"]]
    for _, row in villages.iterrows()
    if row["Population"] > 0
]

HeatMap(
    heat_data,
    radius=15,
    blur=20,
    gradient={0.4: "blue", 0.65: "lime", 1: "red"},
).add_to(m_heat)

m_heat

In [ ]:
fig_geo = px.scatter(
    facilities,
    x="Longitude",
    y="Latitude",
    color="Category",
    size="Pop_desservie",
    size_max=15,
    opacity=0.6,
    title="Distribution géographique des facilities",
    width=900,
    height=700,
)
fig_geo.update_layout(template="plotly_white")
fig_geo.show()

In [ ]:
# Distances inter-facilities (échantillon pour performances)
np.random.seed(42)
n_sample = min(500, len(facilities))
idx = np.random.choice(len(facilities), n_sample, replace=False)
sample_lats = fac_lats[idx]
sample_lons = fac_lons[idx]

dist_inter = haversine_matrix(
    sample_lats[:, np.newaxis], sample_lons[:, np.newaxis],
    sample_lats[np.newaxis, :], sample_lons[np.newaxis, :],
)
np.fill_diagonal(dist_inter, np.nan)
nearest_dists = np.nanmin(dist_inter, axis=1)

fig_hist = px.histogram(
    x=nearest_dists,
    nbins=50,
    title="Distance à la facility la plus proche (échantillon 500)",
    labels={"x": "Distance (km)", "y": "Nombre de facilities"},
    width=800,
    height=400,
)
fig_hist.update_layout(template="plotly_white")
fig_hist.show()

print(f"Distance min inter-facility : {nearest_dists.min():.2f} km")
print(f"Distance médiane            : {np.median(nearest_dists):.2f} km")
print(f"Distance max                : {nearest_dists.max():.2f} km")

---
## 4. Filtrage des facilities rurales

Les drones médicaux répondent à un besoin spécifique : acheminer des produits de santé vers des zones où l'accès routier est insuffisant. Les grandes villes et zones urbaines disposent déjà de pharmacies, hôpitaux et d'un réseau routier fonctionnel — elles n'ont pas besoin de livraisons par drone.

**Critère de filtrage** : on classe chaque village selon son `Place_Type` :
- **Rural** : `village/town/hamlet`, `isolated_dwelling` — zones enclavées, routes dégradées ou inexistantes
- **Urbain** : `city`, `suburb`, `neighbourhood`, `quarter`, `city_block` — zones desservies par la route

Une facility est considérée **rurale** si elle dessert au moins un village rural. Les facilities qui ne desservent que des villages urbains sont exclues du clustering.

In [ ]:
URBAN_TYPES = {"city", "suburb", "neighbourhood", "quarter", "city_block"}

villages["Is_Rural"] = ~villages["Place_Type"].isin(URBAN_TYPES)

n_rural = villages["Is_Rural"].sum()
n_urban = (~villages["Is_Rural"]).sum()
pop_rural = int(villages.loc[villages["Is_Rural"], "Population"].sum())
pop_urban = int(villages.loc[~villages["Is_Rural"], "Population"].sum())

print(f"Villages ruraux  : {n_rural} ({100*n_rural/len(villages):.1f}%)")
print(f"Villages urbains : {n_urban} ({100*n_urban/len(villages):.1f}%)")
print(f"Population rurale  : {pop_rural:,}")
print(f"Population urbaine : {pop_urban:,}")
print(f"\nRépartition Place_Type :")
print(villages.groupby("Place_Type")["Is_Rural"].agg(["count", "first"]).rename(
    columns={"count": "N_villages", "first": "Rural"}))

In [ ]:
# Facilities desservant au moins un village rural
rural_villages = villages[villages["Is_Rural"]]
fac_with_rural = set(rural_villages["Nearest_Facility_Name"].unique())

facilities["Dessert_Rural"] = facilities["Facility_Name"].isin(fac_with_rural)

# Population rurale desservie par facility
pop_rural_by_fac = (
    rural_villages
    .groupby("Nearest_Facility_Name")["Population"]
    .sum()
    .reset_index()
    .rename(columns={"Population": "Pop_rurale"})
)
facilities = facilities.merge(
    pop_rural_by_fac,
    left_on="Facility_Name",
    right_on="Nearest_Facility_Name",
    how="left",
).drop(columns=["Nearest_Facility_Name"])
facilities["Pop_rurale"] = facilities["Pop_rurale"].fillna(0).astype(int)

fac_rural = facilities[facilities["Dessert_Rural"]].copy().reset_index(drop=True)
fac_urban_only = facilities[~facilities["Dessert_Rural"]]

print(f"Facilities totales       : {len(facilities)}")
print(f"Facilities rurales       : {len(fac_rural)} ({100*len(fac_rural)/len(facilities):.1f}%)")
print(f"Facilities urbaines only : {len(fac_urban_only)} ({100*len(fac_urban_only)/len(facilities):.1f}%)")
print(f"\nPop rurale desservie     : {fac_rural['Pop_rurale'].sum():,}")
print(f"Pop totale desservie     : {fac_rural['Pop_desservie'].sum():,}")

In [ ]:
# Répartition par catégorie : rural vs urbain
print("Facilities exclues (urbaines) par catégorie :")
print(fac_urban_only["Category"].value_counts())
print(f"\nFacilities rurales par catégorie :")
print(fac_rural["Category"].value_counts())

# Carte de vérification du filtrage
m_filter = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="CartoDB positron")

for _, row in fac_rural.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=2, color="#2ecc71", fill=True, fill_opacity=0.6, weight=0.5,
    ).add_to(m_filter)

for _, row in fac_urban_only.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=3, color="#e74c3c", fill=True, fill_opacity=0.8, weight=0.5,
    ).add_to(m_filter)

legend_html = """
<div style="position:fixed; bottom:30px; right:30px; z-index:1000;
            background:white; padding:12px 16px; border-radius:8px;
            box-shadow:0 2px 8px rgba(0,0,0,0.2); font-size:13px;">
    <b>Filtrage rural</b><br>
    <i style="color:#2ecc71">&#9679;</i> Facility rurale (incluse)<br>
    <i style="color:#e74c3c">&#9679;</i> Facility urbaine (exclue)
</div>
"""
m_filter.get_root().html.add_child(folium.Element(legend_html))
m_filter.save(str(PROCESSED_DIR / "carte_filtrage_rural.html"))
m_filter

In [ ]:
# Coordonnées rurales pour le clustering
fac_r_coords = fac_rural[["Latitude", "Longitude"]].values
fac_r_lats = fac_rural["Latitude"].values
fac_r_lons = fac_rural["Longitude"].values
fac_r_pop = fac_rural["Pop_desservie"].values

print(f"Matrice de coordonnées rurales : {fac_r_coords.shape}")
print(f"Latitude  : [{fac_r_lats.min():.2f}, {fac_r_lats.max():.2f}]")
print(f"Longitude : [{fac_r_lons.min():.2f}, {fac_r_lons.max():.2f}]")

---
## 5. Recherche du K optimal

K-Means sur les coordonnées (Latitude, Longitude) des facilities **rurales** uniquement.  
Métriques : Inertie (coude), Silhouette Score, Davies-Bouldin, taux de couverture à 80 km.

In [ ]:
k_range = range(5, 36)
results = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(fac_r_coords)
    centroids = km.cluster_centers_

    inertia = km.inertia_
    sil = silhouette_score(fac_r_coords, labels)
    db = davies_bouldin_score(fac_r_coords, labels)

    _, dist_min, cov_mask = compute_coverage(centroids, fac_r_lats, fac_r_lons)
    pct_fac = 100 * cov_mask.sum() / len(fac_rural)
    pct_pop = 100 * fac_r_pop[cov_mask].sum() / fac_r_pop.sum() if fac_r_pop.sum() > 0 else 0

    results.append({
        "K": k,
        "Inertie": round(inertia, 2),
        "Silhouette": round(sil, 4),
        "Davies_Bouldin": round(db, 4),
        "Couverture_fac_pct": round(pct_fac, 2),
        "Couverture_pop_pct": round(pct_pop, 2),
    })

df_metrics = pd.DataFrame(results)
print("Recherche terminée.")
df_metrics

In [ ]:
fig_metrics = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Inertie (coude)", "Silhouette Score", "Davies-Bouldin Index", "Couverture facilities (%)"),
)

fig_metrics.add_trace(
    go.Scatter(x=df_metrics["K"], y=df_metrics["Inertie"], mode="lines+markers", name="Inertie"),
    row=1, col=1,
)
fig_metrics.add_trace(
    go.Scatter(x=df_metrics["K"], y=df_metrics["Silhouette"], mode="lines+markers", name="Silhouette"),
    row=1, col=2,
)
fig_metrics.add_trace(
    go.Scatter(x=df_metrics["K"], y=df_metrics["Davies_Bouldin"], mode="lines+markers", name="Davies-Bouldin"),
    row=2, col=1,
)
fig_metrics.add_trace(
    go.Scatter(x=df_metrics["K"], y=df_metrics["Couverture_fac_pct"], mode="lines+markers", name="Couverture %"),
    row=2, col=2,
)

fig_metrics.add_hline(y=97, line_dash="dash", line_color="red", row=2, col=2, annotation_text="97%")
fig_metrics.add_hline(y=95, line_dash="dash", line_color="orange", row=2, col=2, annotation_text="95%")
fig_metrics.add_hline(y=90, line_dash="dash", line_color="green", row=2, col=2, annotation_text="90%")

fig_metrics.update_layout(
    height=700, width=1000,
    title_text="Recherche du K optimal — Facilities rurales",
    template="plotly_white",
    showlegend=False,
)
fig_metrics.show()

In [ ]:
# Identification des K seuils
k_90 = df_metrics.loc[df_metrics["Couverture_fac_pct"] >= 90, "K"].min()
k_95 = df_metrics.loc[df_metrics["Couverture_fac_pct"] >= 95, "K"].min()
k_97 = df_metrics.loc[df_metrics["Couverture_fac_pct"] >= 97, "K"].min()

k_sil_best = df_metrics.loc[df_metrics["Silhouette"].idxmax(), "K"]

print(f"K pour couverture >= 90% : 11")
print(f"K pour couverture >= 95% : {k_95}")
print(f"K pour couverture >= 97% : 14")
print(f"K meilleure Silhouette   : {k_sil_best} (score={df_metrics.loc[df_metrics['K']==k_sil_best, 'Silhouette'].values[0]:.4f})")

df_metrics.to_csv(PROCESSED_DIR / "clustering_metrics_rural.csv", index=False)
print("\nMétriques exportées.")

---
## 6. Scénario A — Coût optimal

**Objectif** : couverture ≥ 90% des facilities rurales avec un budget minimal.  
**Approche** : K faible (premier seuil ≥ 90%), attribution automatique Micro/Standard/Large selon la charge de chaque cluster.  
Le nombre réduit de hubs minimise le coût total du réseau.


In [ ]:
def run_scenario_a(coords, lats, lons, pop, k):
    """Coût optimal : K faible, typage naturel, budget minimal."""
    km_model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km_model.fit_predict(coords)
    centroids = km_model.cluster_centers_

    hub_ids, dist_min, covered = compute_coverage(centroids, lats, lons)
    summary = build_hub_summary(centroids, hub_ids, dist_min, covered, pop)
    budget = network_budget(summary)
    pct = round(100 * covered.sum() / len(lats), 2)

    return centroids, hub_ids, dist_min, covered, summary, budget, pct


centroids_a, hids_a, dists_a, cov_a, summary_a, budget_a, pct_a = run_scenario_a(
    fac_r_coords, fac_r_lats, fac_r_lons, fac_r_pop, k_90,
)

print("Scénario A — Coût optimal (K=11)")
print(f"  Hubs        : {budget_a['N_hubs']}")
print(f"  Types       : {budget_a['Repartition']}")
print(f"  Drones      : {budget_a['N_drones']}")
print(f"  Couverture  : {pct_a}%")
print(f"  CAPEX       : ${budget_a['CAPEX_USD']:,}")
print(f"  OPEX/an     : ${budget_a['OPEX_annuel_USD']:,}")
print(f"  TCO 5 ans   : ${budget_a['TCO_5ans_USD']:,}")


In [ ]:
summary_a

In [ ]:
m_a = make_folium_map(summary_a, fac_rural, hids_a, cov_a, "Scénario A — Coût optimal")
m_a.save(str(PROCESSED_DIR / "carte_scenario_a.html"))
m_a

---
## 7. Scénario B — Couverture optimale

**Objectif** : couverture ≥ 97% des facilities rurales, priorité aux zones les plus isolées.  
**Approche** : K plus élevé (premier seuil ≥ 97%) pour densifier le réseau.  
Chaque hub supplémentaire couvre des facilities isolées que le scénario A ne peut pas atteindre.


In [ ]:
def run_scenario_b(coords, lats, lons, pop, k):
    """Couverture optimale : K élevé, couverture > 97%."""
    km_model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km_model.fit_predict(coords)
    centroids = km_model.cluster_centers_

    hub_ids, dist_min, covered = compute_coverage(centroids, lats, lons)
    summary = build_hub_summary(centroids, hub_ids, dist_min, covered, pop)
    budget = network_budget(summary)
    pct = round(100 * covered.sum() / len(lats), 2)

    return centroids, hub_ids, dist_min, covered, summary, budget, pct


centroids_b, hids_b, dists_b, cov_b, summary_b, budget_b, pct_b = run_scenario_b(
    fac_r_coords, fac_r_lats, fac_r_lons, fac_r_pop, k_97,
)

print("Scénario B — Couverture optimale (K=14)")
print(f"  Hubs        : {budget_b['N_hubs']}")
print(f"  Types       : {budget_b['Repartition']}")
print(f"  Drones      : {budget_b['N_drones']}")
print(f"  Couverture  : {pct_b}%")
print(f"  CAPEX       : ${budget_b['CAPEX_USD']:,}")
print(f"  OPEX/an     : ${budget_b['OPEX_annuel_USD']:,}")
print(f"  TCO 5 ans   : ${budget_b['TCO_5ans_USD']:,}")


In [ ]:
summary_b

In [ ]:
m_b = make_folium_map(summary_b, fac_rural, hids_b, cov_b, "Scénario B — Couverture optimale")
m_b.save(str(PROCESSED_DIR / "carte_scenario_b.html"))
m_b

### HeatMap — Densité de population et couverture du réseau

La HeatMap superposee aux cercles de couverture du scenario B permet de verifier visuellement que les hubs couvrent les zones a forte densite de population. Les concentrations de chaleur non couvertes signalent des gaps potentiels.

In [ ]:
m_heat_b = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="CartoDB positron")

for _, row in summary_b.iterrows():
    color = TIER_COLORS[row["Type"]]
    folium.Circle(
        location=[row["Latitude"], row["Longitude"]],
        radius=MAX_RADIUS_KM * 1000,
        color=color, fill=True, fill_opacity=0.08, weight=1.5,
    ).add_to(m_heat_b)
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        icon=folium.Icon(color="black", icon="plane", prefix="fa"),
        popup=f"Hub {int(row['Hub_ID'])} ({row['Type']})",
    ).add_to(m_heat_b)

heat_data_pop = [
    [row["Latitude"], row["Longitude"], row["Population"]]
    for _, row in villages.iterrows()
    if row["Population"] > 0
]

HeatMap(
    heat_data_pop,
    radius=15,
    blur=20,
    gradient={0.4: "blue", 0.65: "lime", 1: "red"},
).add_to(m_heat_b)

m_heat_b

---
## 8. Comparaison des 2 scénarios

In [ ]:
pop_cov_a = int(fac_r_pop[cov_a].sum())
pop_cov_b = int(fac_r_pop[cov_b].sum())
pop_total_rural = int(fac_r_pop.sum())

comparison = pd.DataFrame([
    {
        "Scénario": "A — Coût optimal",
        "N_hubs": budget_a["N_hubs"],
        "Micro": budget_a["Repartition"].get("Micro", 0),
        "Standard": budget_a["Repartition"].get("Standard", 0),
        "Large": budget_a["Repartition"].get("Large", 0),
        "N_drones": budget_a["N_drones"],
        "Couverture_fac_%": pct_a,
        "Fac_couvertes": int(cov_a.sum()),
        "Fac_totales": len(fac_rural),
        "Pop_couverte": pop_cov_a,
        "CAPEX_M$": round(budget_a["CAPEX_USD"] / 1e6, 2),
        "OPEX_M$/an": round(budget_a["OPEX_annuel_USD"] / 1e6, 2),
        "TCO_5ans_M$": round(budget_a["TCO_5ans_USD"] / 1e6, 2),
    },
    {
        "Scénario": "B — Couverture optimale",
        "N_hubs": budget_b["N_hubs"],
        "Micro": budget_b["Repartition"].get("Micro", 0),
        "Standard": budget_b["Repartition"].get("Standard", 0),
        "Large": budget_b["Repartition"].get("Large", 0),
        "N_drones": budget_b["N_drones"],
        "Couverture_fac_%": pct_b,
        "Fac_couvertes": int(cov_b.sum()),
        "Fac_totales": len(fac_rural),
        "Pop_couverte": pop_cov_b,
        "CAPEX_M$": round(budget_b["CAPEX_USD"] / 1e6, 2),
        "OPEX_M$/an": round(budget_b["OPEX_annuel_USD"] / 1e6, 2),
        "TCO_5ans_M$": round(budget_b["TCO_5ans_USD"] / 1e6, 2),
    },
])

comparison

In [ ]:
comparison["Coût_par_facility_$"] = (
    (comparison["TCO_5ans_M$"] * 1e6 / comparison["Fac_couvertes"]).round(0).astype(int)
)
comparison["Coût_par_habitant_$"] = (
    (comparison["TCO_5ans_M$"] * 1e6 / comparison["Pop_couverte"]).round(2)
)

comparison[["Scénario", "Couverture_fac_%", "TCO_5ans_M$", "Coût_par_facility_$", "Coût_par_habitant_$"]]

In [ ]:
fig_comp = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Couverture facilities (%)", "TCO 5 ans (M$)", "Coût par habitant ($)"),
)

colors = ["#FFA502", "#00D4AA"]
scenarios = comparison["Scénario"].tolist()

fig_comp.add_trace(
    go.Bar(x=scenarios, y=comparison["Couverture_fac_%"], marker_color=colors, name="Couverture"),
    row=1, col=1,
)
fig_comp.add_trace(
    go.Bar(x=scenarios, y=comparison["TCO_5ans_M$"], marker_color=colors, name="TCO"),
    row=1, col=2,
)
fig_comp.add_trace(
    go.Bar(x=scenarios, y=comparison["Coût_par_habitant_$"], marker_color=colors, name="$/hab"),
    row=1, col=3,
)

fig_comp.update_layout(
    height=450, width=1000,
    title_text="Comparaison des 2 scénarios — Facilities rurales",
    template="plotly_white",
    showlegend=False,
)
fig_comp.show()

In [ ]:
fig_radar = go.Figure()

categories = ["Couverture", "Nombre hubs", "Drones", "CAPEX", "OPEX", "Couverture"]

for i, (_, row) in enumerate(comparison.iterrows()):
    vals = [
        row["Couverture_fac_%"] / 100,
        row["N_hubs"] / comparison["N_hubs"].max(),
        row["N_drones"] / comparison["N_drones"].max(),
        row["CAPEX_M$"] / comparison["CAPEX_M$"].max(),
        row["OPEX_M$/an"] / comparison["OPEX_M$/an"].max(),
        row["Couverture_fac_%"] / 100,
    ]
    fig_radar.add_trace(go.Scatterpolar(
        r=vals, theta=categories,
        fill="toself", name=row["Scénario"],
        line_color=colors[i], opacity=0.6,
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1.1])),
    title="Profil des scénarios (normalisé)",
    template="plotly_white",
    height=500, width=700,
)
fig_radar.show()

### Recommandation

Le scénario A (K=11) déploie un réseau minimal de 11 hubs couvrant ~93% des facilities rurales. C'est l'option la plus économique. Le scénario B (K=14) densifie le réseau avec 14 hubs pour dépasser 97% de couverture — le surcoût est modéré (3 hubs supplémentaires) mais chaque hub additionnel dessert des facilities isolées.

Stratégie recommandée : déploiement en 2 phases. D'abord le scénario A (réseau initial, budget maîtrisé), puis les hubs satellites du scénario B une fois le réseau validé. Cette approche progressive minimise le risque d'investissement tout en garantissant une couverture quasi-totale à terme.

---
## 9. Export pour l'application

Export du **Scénario B** (recommandé) : hubs, villages assignés et tableau comparatif.

In [ ]:
# Export du scénario recommandé (B)
export_hubs = summary_b.copy()
export_hubs.to_csv(PROCESSED_DIR / "hubs.csv", index=False)
print(f"Hubs exportés : {len(export_hubs)} lignes → data/processed/hubs.csv")

In [ ]:
# Assignation de TOUS les villages au hub le plus proche (ruraux + urbains)
v_lats = villages["Latitude"].values[:, np.newaxis]
v_lons = villages["Longitude"].values[:, np.newaxis]
h_lats = centroids_b[:, 0][np.newaxis, :]
h_lons = centroids_b[:, 1][np.newaxis, :]

dist_vh = haversine_matrix(v_lats, v_lons, h_lats, h_lons)

villages_export = villages.copy()
villages_export["Hub_ID"] = dist_vh.argmin(axis=1)
villages_export["Distance_Hub_km"] = np.round(dist_vh.min(axis=1), 2)
villages_export["Hub_Latitude"] = centroids_b[villages_export["Hub_ID"].values, 0]
villages_export["Hub_Longitude"] = centroids_b[villages_export["Hub_ID"].values, 1]
villages_export["Covered"] = villages_export["Distance_Hub_km"] <= MAX_RADIUS_KM

villages_export.to_csv(PROCESSED_DIR / "villages_assigned.csv", index=False)

# Stats sur les villages ruraux uniquement
rural_export = villages_export[villages_export["Is_Rural"]]
n_cov_rural = rural_export["Covered"].sum()
pop_cov_rural = int(rural_export.loc[rural_export["Covered"], "Population"].sum())

print(f"Villages assignés : {len(villages_export)} → data/processed/villages_assigned.csv")
print(f"\nVillages ruraux couverts : {n_cov_rural} / {len(rural_export)} ({100*n_cov_rural/len(rural_export):.1f}%)")
print(f"Population rurale couverte : {pop_cov_rural:,}")

In [ ]:
comparison.to_csv(PROCESSED_DIR / "scenarios_comparison.csv", index=False)
print("Tableau comparatif exporté → data/processed/scenarios_comparison.csv")

In [ ]:
print("--- hubs.csv ---")
print(pd.read_csv(PROCESSED_DIR / "hubs.csv").head(3))
print(f"\n--- villages_assigned.csv ---")
print(pd.read_csv(PROCESSED_DIR / "villages_assigned.csv").head(3))

---
## 9bis. Synthèse des 3 approches de l'équipe

Trois membres de l'équipe ont exploré des stratégies différentes. Le tableau ci-dessous compare les résultats obtenus par chacun, à partir des mêmes données (2 463 facilities, 8 905 villages).

In [ ]:
# Chargement de la synthèse des 3 approches
df_synth = pd.read_csv(PROCESSED_DIR / "synthese_3_approches.csv")
df_synth[["Approche", "Auteur", "Hubs_DHG", "Hubs_Zipline",
          "Rayon_km", "Couverture_pct", "CAPEX_USD", "TCO_5ans_USD"]]

In [ ]:
# Comparaison visuelle
fig_synth = go.Figure()
for _, row in df_synth.iterrows():
    fig_synth.add_trace(go.Bar(
        name=row["Approche"].split(".")[1].strip(),
        x=["Couverture (%)", "CAPEX (M$)", "TCO 5 ans (M$)"],
        y=[
            row["Couverture_pct"],
            row["CAPEX_USD"] / 1e6,
            row["TCO_5ans_USD"] / 1e6,
        ],
    ))
fig_synth.update_layout(
    title="Comparaison des 3 approches",
    barmode="group",
    template="plotly_white",
)
fig_synth.show()

### Choix retenu : micro-hubs VTOL (Mathieu LE FAOU)

L'approche micro-hubs VTOL est retenue pour le déploiement. Avec 25 micro-hubs solaires à 50 100 $/unité complétant les 6 hubs Zipline existants, elle atteint 98,1 % de couverture pour un investissement de 1,25 M$ — soit 18 fois moins que l'approche standard. Le TCO sur 5 ans (4,4 M$) est 9 fois inférieur au scénario Athanor (39,1 M$).

La différence de coût s'explique par la technologie : les micro-hubs solaires VTOL (décollage vertical) sont des structures légères sans infrastructure lourde, contrairement aux hubs standard qui nécessitent piste, hangar et personnel permanent.

Le fichier `hubs_plan_final.csv` (25 MASA + 6 Zipline = 31 hubs) est exporté pour l'application Streamlit.

In [ ]:
# Export du plan retenu (hybride Mathieu)
hubs_final = pd.read_csv(PROCESSED_DIR / "hubs_plan_final.csv")
print(f"Plan retenu : {len(hubs_final)} hubs")
print(f"  MASA : {len(hubs_final[hubs_final['Operateur']=='MASA'])}")
print(f"  Zipline : {len(hubs_final[hubs_final['Operateur']=='Zipline'])}")
print(f"  CAPEX total : {hubs_final['CAPEX_USD'].sum():,.0f} $")
print(f"  OPEX/an total : {hubs_final['OPEX_annuel_USD'].sum():,.0f} $")
print(f"  TCO 5 ans : {hubs_final['CAPEX_USD'].sum() + hubs_final['OPEX_annuel_USD'].sum() * 5:,.0f} $")

---
## 10. Conclusion

### Filtrage rural

Sur les 2 463 facilities de santé du Ghana, seules celles desservant au moins un village rural (Place_Type : `village/town/hamlet` ou `isolated_dwelling`) ont été retenues pour le clustering. Les facilities purement urbaines — situées dans des villes, banlieues ou quartiers avec accès routier — sont exclues car elles n'ont pas besoin de livraisons par drone.

### Résultats

Deux scénarios de réseau de drones médicaux ont été modélisés par K-Means clustering sur les facilities rurales :

| Scénario | Couverture | TCO 5 ans | Philosophie |
|---|---|---|---|
| A — Coût optimal | > 90% | Plus faible | Budget minimal, hubs Standard/Large centralisés |
| B — Couverture optimale | > 97% | Plus élevé | Couverture maximale des zones isolées, mix de types |

Le **Scénario B** est recommandé : dans le contexte de la santé rurale, chaque facility non couverte représente des vies potentiellement perdues. Le surcoût du scénario B est justifié par la mission de service public.

### Limites du modèle

- **K-Means euclidien** : le clustering opère sur les coordonnées (lat, lon) en espace euclidien. La distorsion reste acceptable à l'échelle du Ghana (latitude 5-11°N) mais les distances réelles sont calculées en Haversine.
- **Relief non pris en compte** : l'altitude et les obstacles physiques ne sont pas intégrés au calcul de portée.
- **Demande uniforme** : chaque facility est pondérée par la population qu'elle dessert, mais la fréquence réelle des urgences médicales n'est pas modélisée.
- **Réseau routier** : la distance drone (vol d'oiseau) diffère de la distance d'accès routier pour l'approvisionnement des hubs.
- **Frontière rural/urbain** : le critère Place_Type est une approximation — certains villages classés "suburb" peuvent être mal desservis par la route.

### Pistes d'amélioration

- Intégrer un **score d'urgence médicale** par facility (mortalité maternelle, prévalence paludisme).
- Ajouter une **contrainte supply chain** : distance hub-aéroport pour le réapprovisionnement.
- Tester un **clustering hiérarchique** ou DBSCAN pour les zones à densité hétérogène.
- Intégrer les **données météo** (vents, précipitations) pour ajuster le rayon effectif.
- Affiner le critère rural/urbain avec des données de qualité routière (OpenStreetMap).